# cudf-bench: string-column probe (Step 3b)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. ~10 min. **Click Allow on the download at the end.**

Finding under test: cuDF `groupby_agg` at skew 1.5 was 65% slower **only when the table carried an unused string column**. This probe isolates it — same groupby, same data, string columns dialed 0/1/2, each config in its own fresh process. Polars runs the same 2×2 as a control (effect should be cuDF-specific).

Predictions if string-column hypothesis is right: `str_cols=0` rows flat across skew; `str_cols=1` slow at skew 1.5; `str_cols=2` slower still (dose–response). Polars unaffected.

In [ ]:
!nvidia-smi -L

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
!git pull -q
!rm -f results/probe.csv
%pip install -q -e .

## cuDF: str_cols × skew (each line = fresh process, 7 reps)

In [ ]:
for sc in [0, 1, 2]:
    for sk in ["0", "1.5"]:
        !python -m benchmark.runner --backend cudf --ops groupby_agg --rows 1e7 --skew {sk} --str-cols {sc} --reps 7 --out results/probe.csv

## Polars control: same 2×2

In [ ]:
for sc in [0, 1]:
    for sk in ["0", "1.5"]:
        !python -m benchmark.runner --backend polars --ops groupby_agg --rows 1e7 --skew {sk} --str-cols {sc} --reps 7 --out results/probe.csv

## Quick look

In [ ]:
import pandas as pd
df = pd.read_csv("results/probe.csv")
print(df.pivot_table(index=["backend", "str_cols"], columns="skew", values="median_s").round(4))

## Auto-download (click Allow)

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/probe.csv")